# Weather Data Source Comparison
### CS611 MLE Group Project — Dengue Outbreak Risk Prediction

Compares:
- **MSS file** (`mss_weather_2013_2016.csv`) — scraped from weather.gov.sg, ~100 stations
- **NEA file** (`nea_weather_2017_2020.csv`) — from NEA v2 API, ~20 stations

Goal: understand coverage gaps and column differences before merging.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DATA_DIR = os.path.expanduser("~/Desktop/CS611_dengue/data/dengue")

MSS_FILE = os.path.join(DATA_DIR, "mss_weather_2013_2016.csv")
NEA_FILE = os.path.join(DATA_DIR, "nea_weather.csv")

mss = pd.read_csv(MSS_FILE)
nea = pd.read_csv(NEA_FILE)

print(f"MSS shape: {mss.shape}")
print(f"NEA shape: {nea.shape}")

## 1. Column Comparison
What columns exist in each file, and do they share names?

In [ ]:
mss_cols = set(mss.columns.str.strip())
nea_cols = set(nea.columns.str.strip())

shared    = mss_cols & nea_cols
mss_only  = mss_cols - nea_cols
nea_only  = nea_cols - mss_cols

print("=" * 50)
print(f"Shared columns ({len(shared)}):")
for c in sorted(shared): print(f"  ✓  {c}")

print(f"\nMSS only ({len(mss_only)}):")
for c in sorted(mss_only): print(f"  →  {c}")

print(f"\nNEA only ({len(nea_only)}):")
for c in sorted(nea_only): print(f"  →  {c}")

## 2. Station Coverage
How many stations in each, and which ones overlap?

In [ ]:
# Normalise station name column — MSS uses 'Station', NEA may vary
# Adjust 'station_name' below if your NEA file uses a different column
MSS_STATION_COL = "Station"
NEA_STATION_COL = "station_name"  # change if needed

mss_stations = set(mss[MSS_STATION_COL].str.strip().unique())
nea_stations = set(nea[NEA_STATION_COL].str.strip().unique())

overlap   = mss_stations & nea_stations
mss_extra = mss_stations - nea_stations
nea_extra = nea_stations - mss_stations

print(f"MSS stations : {len(mss_stations)}")
print(f"NEA stations : {len(nea_stations)}")
print(f"Overlapping  : {len(overlap)}")
print(f"MSS only     : {len(mss_extra)}")
print(f"NEA only     : {len(nea_extra)}")

print("\nOverlapping stations:")
for s in sorted(overlap): print(f"  ✓  {s}")

print("\nNEA-only stations (won't have 2013-2016 data):")
for s in sorted(nea_extra): print(f"  !  {s}")

## 3. Temporal Coverage
Confirm date ranges and spot any gaps within each file.

In [ ]:
# Build a date column for MSS (has Year, Month, Day columns)
mss["date"] = pd.to_datetime(
    mss[["Year", "Month", "Day"]].rename(columns={"Year":"year","Month":"month","Day":"day"})
)

# NEA file should already have a date column — adjust if needed
nea["date"] = pd.to_datetime(nea["date"])

print("MSS date range:", mss["date"].min().date(), "→", mss["date"].max().date())
print("NEA date range:", nea["date"].min().date(), "→", nea["date"].max().date())

# Check for internal gaps in MSS (missing months)
mss_months = mss.groupby(pd.Grouper(key="date", freq="ME"))["date"].count().reset_index(name="rows")
gap_months = mss_months[mss_months["rows"] == 0]
if gap_months.empty:
    print("\nMSS: no missing months ✓")
else:
    print(f"\nMSS: {len(gap_months)} months with no data:")
    print(gap_months)

## 4. Null Rate by Column
How complete is each dataset? Important for deciding which columns are safe to use.

In [ ]:
def null_report(df, label):
    nulls = df.isnull().mean().mul(100).round(1).sort_values(ascending=False)
    print(f"\n{'─'*40}")
    print(f"  {label} — null rate per column (%)")
    print(f"{'─'*40}")
    for col, pct in nulls.items():
        bar = '█' * int(pct / 5)
        flag = " ⚠️" if pct > 20 else ""
        print(f"  {col:<45} {pct:5.1f}%  {bar}{flag}")

null_report(mss, "MSS 2013–2016")
null_report(nea, "NEA 2017–2020")

## 5. Value Distribution Comparison (Overlapping Stations)
For stations that appear in both datasets, compare distributions of key weather variables.
This is the best sanity check — the distributions should be broadly similar since they're measuring the same city.

In [ ]:
# ── Define the column mapping between MSS and NEA ──────────────────────────
# Adjust right-hand values if your NEA file uses different column names
COL_MAP = {
    "Daily Rainfall Total (mm)" : "rainfall_total_mm",
    "Mean Temperature (°C)"     : "temp_mean_c",
    "Maximum Temperature (°C)"  : "temp_max_c",
    "Minimum Temperature (°C)"  : "temp_min_c",
    "Mean Wind Speed (km/h)"    : "wind_mean_kmh",
}

# Filter to overlapping stations only
mss_overlap = mss[mss[MSS_STATION_COL].isin(overlap)]
nea_overlap = nea[nea[NEA_STATION_COL].isin(overlap)]

available_pairs = [
    (mss_col, nea_col)
    for mss_col, nea_col in COL_MAP.items()
    if mss_col in mss.columns and nea_col in nea.columns
]

if not available_pairs:
    print("No matching columns found — update COL_MAP above to match your NEA column names.")
    print("NEA columns:", list(nea.columns))
else:
    fig, axes = plt.subplots(1, len(available_pairs), figsize=(5 * len(available_pairs), 4))
    if len(available_pairs) == 1:
        axes = [axes]

    for ax, (mss_col, nea_col) in zip(axes, available_pairs):
        mss_vals = mss_overlap[mss_col].dropna()
        nea_vals = nea_overlap[nea_col].dropna()

        ax.hist(mss_vals, bins=40, alpha=0.6, label="MSS 2013–16", color="steelblue")
        ax.hist(nea_vals, bins=40, alpha=0.6, label="NEA 2017–20", color="coral")
        ax.set_title(mss_col.replace(" (", "\n("), fontsize=9)
        ax.legend(fontsize=8)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    plt.suptitle("Distribution Comparison — Overlapping Stations Only", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, "weather_distribution_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved.")

## 6. Summary Stats — Key Variables

In [ ]:
print("MSS — key variable stats:")
mss_stat_cols = [c for c in COL_MAP.keys() if c in mss.columns]
display(mss[mss_stat_cols].describe().round(2))

print("\nNEA — key variable stats:")
nea_stat_cols = [c for c in COL_MAP.values() if c in nea.columns]
display(nea[nea_stat_cols].describe().round(2))

## 7. Decision Checklist

Use this to decide how to handle the mismatch before merging:

| Question | Answer | Action |
|---|---|---|
| Do overlapping stations have similar distributions? | (check plots above) | If yes → safe to concatenate |
| Which columns exist in both? | (check Section 1) | Keep only shared cols for the merged file |
| Are MSS-only stations geographically important? | (check map) | If yes → keep them; fill NEA period with NaN |
| Are NEA-only stations geographically important? | (check map) | If yes → keep them; fill MSS period with NaN |
| Is the null rate for any key column > 30%? | (check Section 4) | Drop that column or impute with caution |

**Recommended merge strategy:**
1. Standardise column names across both files
2. `pd.concat([mss_clean, nea_clean])` — rows from both, union of stations
3. For stations missing in one period, rows simply won't exist (no fake data)
4. In preprocessing, aggregate to Singapore-wide daily mean across all available stations — this naturally handles the coverage difference